# Inference Demo — Automatic Speech Assessment

Load a trained Qwen2-Audio checkpoint, run inference on test audio files,
**listen** to the audio, and inspect the model's responses side-by-side.

In [ ]:
from pathlib import Path

import IPython.display as ipd
import pandas as pd

from asa.inference import load_model, run_inference
from asa.data import TARGET_SR

## 1 — Configuration

Point these at your checkpoint and audio directory.  
Adjust `MAX_FILES` to control how many samples to evaluate.

In [ ]:
MODEL_DIR = Path("../results/sft")
AUDIO_DIR = Path("../data/raw/NISQA_Corpus/NISQA_TEST_FOR/deg")
MAX_FILES = 5

audio_files = sorted(AUDIO_DIR.glob("*.wav"))[:MAX_FILES]
print(f"Found {len(audio_files)} audio files")
for f in audio_files:
    print(f"  • {f.name}")

## 2 — Load model

In [ ]:
processor, model, device = load_model(MODEL_DIR)
print(f"Model loaded on {device}")

## 3 — Run inference

In [ ]:
responses = run_inference(
    model, processor, audio_files,
    device=device,
    max_new_tokens=150,
)

results = pd.DataFrame({
    "file": [f.name for f in audio_files],
    "response": [r.strip() for r in responses],
})
results

## 4 — Listen & compare

Play each audio clip alongside the model's assessment.

In [ ]:
from asa.data import load_audio

for i, row in results.iterrows():
    audio_path = audio_files[i]
    waveform = load_audio(str(audio_path), target_sr=TARGET_SR)

    print(f"\n{'='*60}")
    print(f"File: {row['file']}")
    print(f"Response: {row['response']}")
    print(f"{'='*60}")
    ipd.display(ipd.Audio(waveform, rate=TARGET_SR))